In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!nvidia-smi

Tue Aug  5 02:31:58 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!git clone https://github.com/ziye2chen/LLMs-for-Mathematical-Analysis.git

Cloning into 'LLMs-for-Mathematical-Analysis'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 84 (delta 34), reused 53 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 7.71 MiB | 22.10 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Filtering content: 100% (4/4), 493.70 MiB | 63.22 MiB/s, done.


In [5]:
!pip install --upgrade pip
!pip install faiss-cpu sentence-transformers
!pip install unsloth transformers trl datasets pandas matplotlib rich scipy sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 63.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 43.9 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 39.1 MB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 73.5 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 70.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 68.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 67.7 MB/s  0:00:03
   ━━━━━

In [7]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [8]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [9]:
import torch
print("Pytorch version：")
print(torch.__version__)
print("CUDA Version: ")
print(torch.version.cuda)
print("cuDNN version is :")
print(torch.backends.cudnn.version())

Pytorch version：
2.7.1+cu126
CUDA Version: 
12.6
cuDNN version is :
90501


In [10]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 10240 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",          # Phi-3 2x faster!d
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Math-7B-bnb-4bit", # "unsloth/Meta-Llama-3.1-8B"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

==((====))==  Unsloth 2025.8.1: Fast Qwen2 patching. Transformers: 4.54.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: unsloth/Qwen2.5-Math-7B-bnb-4bit can only handle sequence lengths of at most 4096.
But with kaiokendev's RoPE scaling of 2.5, it can be magically be extended to 10240!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.8.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Classifier

In [12]:
alpaca_prompt = """Là trợ lý toán học, bạn cần phân tích bài toán để tìm ra loại bài toán đó trong Phân tích Thực tế. Cung cấp Dạng bài toán và Kiến thức có thể được sử dụng để giải bài toán này.

### Đề bài:
{}

### Dạng bài toán:
{}

### Kiến thức:
{}"""

In [14]:
EOS_TOKEN = tokenizer.eos_token  # Phải có để tránh model sinh vô hạn

def formatting_prompts_func(examples):
    Problem_Type    = examples["problem_type"]
    Problem       = examples["problem"]
    Knowledge  = examples["knowledge"]
    texts = []
    for problem, problem_type, knowledge in zip(Problem, Problem_Type, Knowledge):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(problem, problem_type, knowledge) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset('csv',data_files = '/content/train_data_with_knowledge.csv', split='train')
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # dataset -> tokenized_train_dataset
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2,

        # Use num_train_epochs = 1, warmup_ratio for full training runs!
        warmup_steps = 5,
        max_steps = 300, # 60 -> 10

        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Unsloth: Tokenizing ["text"]:   0%|          | 0/325 [00:00<?, ? examples/s]

In [16]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.705 GB of memory reserved.


In [17]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 325 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,2.439300
2,2.490200
3,2.213700
4,2.499500
5,2.328300
6,2.352200
7,1.749600
8,1.549300
9,1.419000
10,1.257300


Unsloth: Will smartly offload gradients to save VRAM!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [18]:
model.save_pretrained("MathAnalysis_Qwen_Classifier") # Local saving
tokenizer.save_pretrained("MathAnalysis_Qwen_Classifier")
model.save_pretrained("/content/drive/MyDrive/MathAnalysis_Qwen_Classifier") # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/MathAnalysis_Qwen_Classifier")

('/content/drive/MyDrive/MathAnalysis_Qwen_Classifier/tokenizer_config.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_Classifier/special_tokens_map.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_Classifier/vocab.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_Classifier/merges.txt',
 '/content/drive/MyDrive/MathAnalysis_Qwen_Classifier/added_tokens.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_Classifier/tokenizer.json')

In [19]:
test_prompt = """Là trợ lý toán học, bạn cần phân tích bài toán để tìm ra loại bài toán đó trong Phân tích Thực tế. Cung cấp Dạng bài toán và Kiến thức có thể được sử dụng để giải bài toán này.

### Đề bài:
{}
"""

In [ ]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "MathAnalysis_Qwen_Classifier", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    test_prompt.format(
        "Cho số thực dương \( a \) thỏa mãn \( a^3 = 6(a + 1) \). Chứng minh rằng phương trình \( x^2 + ax + a^2 - 6 = 0 \) là vô nghiệm.", # Problem
    )
], return_tensors = "pt").to("cuda")


==((====))==  Unsloth 2025.8.1: Fast Qwen2 patching. Transformers: 4.54.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: unsloth/Qwen2.5-Math-7B-bnb-4bit can only handle sequence lengths of at most 4096.
But with kaiokendev's RoPE scaling of 2.5, it can be magically be extended to 10240!


In [ ]:
# Tự tính max_new_tokens an toàn
input_len = inputs.input_ids.shape[-1]
max_total_length = max_seq_length  # ví dụ: 10240
safe_max_new_tokens = max_total_length - input_len

# In thử kiểm tra
print(f"Input tokens: {input_len}, Max new tokens: {safe_max_new_tokens}")

# Dùng TextStreamer để in ra từng dòng sinh token
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# Sinh kết quả
_ = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    streamer=text_streamer,
    max_new_tokens=safe_max_new_tokens,
    pad_token_id=tokenizer.eos_token_id
)

Input tokens: 79, Max new tokens: 10161
To solve the problem, we need to analyze the expression \( n \sin(2n! e \pi) \) as \( n \to \infty \). The key to solving this problem lies in understanding the behavior of the sine function when its argument is a large multiple of \( \pi \). Specifically, we need to use the fact that \( e \) can be expressed as an infinite series:

\[ e = \sum_{k=0}^{\infty} \frac{1}{k!} = 1 + \frac{1}{1!} + \frac{1}{2!} + \frac{1}{3!} + \cdots \]

When we multiply \( e \) by \( 2n! \pi \), we get:

\[ 2n! e \pi = 2n! \pi \left(1 + \frac{1}{1!} + \frac{1}{2!} + \cdots + \frac{1}{n!} + \frac{1}{(n+1)!} + \cdots \right) \]

The first \( n+1 \) terms of this series are all integer multiples of \( \pi \), so they do not affect the sine function. The remaining terms give us:

\[ 2n! e \pi = 2n! \pi \left(\frac{1}{n+1} + \frac{1}{(n+1)(n+2)} + \cdots \right) \]

As \( n \to \infty \), the expression inside the sine function can be approximated by:

\[ 2n! e \pi \appro

# Problem Solver

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",          # Phi-3 2x faster!d
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Math-7B-bnb-4bit", # "unsloth/Meta-Llama-3.1-8B"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = "auto",
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

==((====))==  Unsloth 2025.7.3: Fast Qwen2 patching. Transformers: 4.53.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = False,  # ← TẮT ở đây
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
alpaca_prompt = """Với tư cách là trợ lý toán học, hãy giải bài toán sau. Đưa ra lời giải chi tiết, từng bước bằng lập luận toán học chặt chẽ. Nếu bài toán yêu cầu sử dụng phương pháp $\epsilon$-$\delta$ (ví dụ: khi chứng minh giới hạn hoặc tính liên tục), hãy đảm bảo bạn áp dụng đúng phương pháp. Sử dụng ngôn ngữ toán học và ký hiệu chính xác trong suốt bài giải.

### Dạng bài toán:
{}

### Đề bài:
{}

### Kiến thức:
{}

### Lời giải:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    Problem_Type    = examples["problem_type"]
    Problem       = examples["problem"]
    Knowledge = examples["knowledge"]
    Solution  = examples["solution"]
    texts = []
    for problem_type, problem, knowledge, solution in zip(Problem_Type, Problem, Knowledge, Solution):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(problem_type, problem, knowledge, solution) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset('csv',data_files = '/content/train_data_with_knowledge.csv', split='train')
dataset = dataset.map(formatting_prompts_func, batched = True,)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

In [ ]:
len(dataset)

49

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # dataset -> tokenized_train_dataset
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2,

        # Use num_train_epochs = 1, warmup_ratio for full training runs!
        warmup_steps = 5,
        max_steps = 300, # 60 -> 10

        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Unsloth: Tokenizing ["text"]:   0%|          | 0/49 [00:00<?, ? examples/s]

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.824 GB of memory reserved.


In [ ]:

trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49 | Num Epochs = 12 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.836000
2,0.880100
3,1.096800
4,0.849800
5,0.901700
6,0.937600
7,0.825000
8,0.840700
9,0.653300
10,0.745700


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
model.save_pretrained("/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver") # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver")
model.save_pretrained("/content/drive/MyDrive/MathAnalysis_Qwen_Classifier") # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/MathAnalysis_Qwen_Classifier")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver/tokenizer_config.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver/special_tokens_map.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver/vocab.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver/merges.txt',
 '/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver/added_tokens.json',
 '/content/drive/MyDrive/MathAnalysis_Qwen_ProblemSolver/tokenizer.json')

In [ ]:
problem_prompt = """Với tư cách là trợ lý toán học, hãy giải bài toán sau. Đưa ra lời giải chi tiết, từng bước bằng lập luận toán học chặt chẽ. Nếu bài toán yêu cầu sử dụng phương pháp $\epsilon$-$\delta$ (ví dụ: khi chứng minh giới hạn hoặc tính liên tục), hãy đảm bảo bạn áp dụng đúng phương pháp. Sử dụng ngôn ngữ toán học và ký hiệu chính xác trong suốt bài giải.

### Dạng bài toán:
{}

### Đề bài:
{}

### Kiến thức:
{}

### Lời giải:
{}"""

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "MathAnalysis_Qwen_ProblemSolver",  # Đổi sang mô hình ProblemSolver
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)  # Tăng tốc inference

# Nhập bài toán cụ thể
inputs = tokenizer(
[
    problem_prompt.format(
        "Tìm tất cả các giá trị thực của \( a \) và \( b \) sao cho: \[ (a^2 + 1)(b^2 + 1) = (a + 1)(b + 1)(ab + 1) \]",  # Câu hỏi LaTeX hoặc plain text
        "Giải phương trình", # Problem_Type
    )
], return_tensors = "pt").to("cuda")

==((====))==  Unsloth 2025.7.3: Fast Qwen2 patching. Transformers: 4.53.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
# Tính độ dài đầu vào an toàn
input_len = inputs.input_ids.shape[-1]
max_total_length = max_seq_length  # ví dụ: 10240
safe_max_new_tokens = max_total_length - input_len

print(f"Input tokens: {input_len}, Max new tokens: {safe_max_new_tokens}")

# Stream kết quả trả ra từng dòng
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# Sinh lời giải
_ = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    streamer=text_streamer,
    max_new_tokens=safe_max_new_tokens,
    pad_token_id=tokenizer.eos_token_id
)

Input tokens: 126, Max new tokens: 3970
### Solution:
For any $oldsymbol{epsilon} > 0$, we can take positive integers $N_1$ and $N_2$ such that
\[
|a_n - lpha| < oldsymbol{epsilon}
\]
for any $n > N_1$ and that
\[
|b_n - eta| < oldsymbol{epsilon}
\]
for any $n > N_2$. Put $N = \max\{N_1, N_2\}$. Then both $|a_n - lpha|$ and $|b_n - eta|$ are less than $oldsymbol{epsilon}$ for any $n > N$. Now we divide the sum into two parts as follows:
\[
\frac{a_0 b_n + a_1 b_{n-1} + \cdots + a_n b_0}{n} = \frac{a_0 b_n + \cdots + a_N b_{n-N}}{n} + \frac{a_{N+1} b_{n-N-1} + \cdots + a_n b_0}{n}.
\]

For the first fraction on the right-hand side, we have
\[
\left| \frac{a_0 b_n + \cdots + a_N b_{n-N}}{n} - lpha eta \right| \leq \frac{|a_0 (b_n - eta)| + \cdots + |a_N (b_{n-N} - eta)|}{n} + \frac{|b_0 (a_0 - lpha)| + \cdots + |b_{n-N} (a_N - lpha)|}{n}.
\]

Both $|a_0|, \ldots, |a_N|$ and $|b_0|, \ldots, |b_{n-N}|$ are bounded by some constants $K_1$ and $K_2$ respectively. Hence the righ